In [0]:
import csv, io, os, requests
from datetime import datetime, timedelta, timezone

CSV_BASE = "https://opendata.fcc.gov/resource/3xyp-aqkj.csv"
PAGE_SIZE = 50000
LOOKBACK_DAYS = 5
SILVER = "robocall.bronze.fcc_complaints_silver"

dbutils.widgets.text("app_token", "", "Socrata app token (optional)")
dbutils.widgets.dropdown("mode", "incremental", ["incremental", "full"], "Load mode")
app_token = dbutils.widgets.get("app_token").strip()
mode = dbutils.widgets.get("mode")

# --- determine watermark -------------------------------------------------
watermark = None
if mode == "incremental" and spark.catalog.tableExists(SILVER):
    row = spark.sql(f"SELECT max(created_at) AS hw FROM {SILVER}").first()
    if row and row.hw:
        watermark = (row.hw - timedelta(days=LOOKBACK_DAYS)).strftime("%Y-%m-%dT%H:%M:%S.000")

print(f"mode={mode}  watermark={watermark or '(none — full load)'}")

# --- fetch ---------------------------------------------------------------
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
VOLUME = f"/Volumes/robocall/source_data/fcc_raw/{run_id}"

session = requests.Session()
if app_token:
    session.headers["X-App-Token"] = app_token

offset = part = total = 0

while True:
    params = {"$limit": PAGE_SIZE, "$offset": offset, "$order": ":id"}
    if watermark:
        params["$where"] = f"date_created > '{watermark}'"

    print(f"fetching rows {offset:,} .. {offset + PAGE_SIZE - 1:,}")
    resp = session.get(CSV_BASE, params=params, timeout=180)
    resp.raise_for_status()

    rows = sum(1 for _ in csv.reader(io.StringIO(resp.text))) - 1
    if rows <= 0:
        print("no rows returned, stopping")
        break

    os.makedirs(VOLUME, exist_ok=True)   # only create once we have data
    with open(os.path.join(VOLUME, f"part_{part:04d}.csv"), "w", encoding="utf-8") as fh:
        fh.write(resp.text)

    total += rows
    part += 1
    print(f"  wrote part_{part - 1:04d} ({rows:,} rows)")

    if rows < PAGE_SIZE:
        break
    offset += PAGE_SIZE

if total == 0:
    print("nothing new — no folder created")
else:
    print(f"done: {total:,} rows across {part} files → {VOLUME}")